# 3.6 — Catalyst, AQE and Skew

**Chapter 3, sections 3.5, 3.8, 3.9.2 and 3.10**, and Exercises 3, 5, 6, 8 and 9.

**The question this notebook answers:** the chapter claims that a DataFrame query is not
executed as written. What does Spark actually run, how do you find out, and what does it change
its mind about while the query is in flight?

Five things get demonstrated rather than asserted:

1. **The four representations.** `explain(mode="extended")` prints the parsed, analyzed,
   optimized and physical plans side by side, so the Catalyst pipeline of section 3.8 is
   visible in one output.
2. **SQL and the method chain are the same query.** Their physical plans are compared
   character by character (§3.5, Exercise 6).
3. **The join strategy is chosen by size.** Broadcast hash below
   `autoBroadcastJoinThreshold`, sort-merge above it, and the `broadcast()` hint overriding a
   mis-estimate (§3.9.2, Exercise 5).
4. **AQE re-optimizes at runtime.** Two hundred shuffle partitions coalesced to fit the
   result, and a planned sort-merge join converted to a broadcast join *after* the shuffle
   revealed one side to be small (§3.10, Exercise 9).
5. **The "ninety-nine percent complete" problem.** A skewed join built on purpose, the
   straggler measured, and then AQE's skew split curing it (§3.10.1, Exercise 8).

All data is generated in the notebook from fixed seeds. Runs on a laptop in about two minutes.
**Timings are machine-dependent**; the plans and the partition counts are not.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
WORK = os.path.join(SCRATCH, "ch03-catalyst")
os.makedirs(WORK, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.6")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")   # keep printed output clean
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


In [2]:
import re

def final_plan(df):
    """The plan Spark *actually ran*, after AQE has finished revising it.
    `explain()` prints the plan as it stood before execution; this prints the plan as
    it ended up, which is where every AQE decision shows.  Call it after an action."""
    text = df._jdf.queryExecution().executedPlan().toString()
    return text.split("== Initial Plan ==")[0].rstrip()

def normalise(plan):
    """Strip Catalyst's per-expression identifiers (`fare#3`, `plan_id=41`) so that two
    plans built from different DataFrame objects can be compared for structure."""
    return re.sub(r"plan_id=\d+", "plan_id", re.sub(r"#\d+L?", "#", plan))

def sql_metrics():
    """Named SQL metrics of the query that just ran -- the numbers on the SQL tab of the UI."""
    store = spark._jsparkSession.sharedState().statusStore()
    execs = store.executionsList()
    last = execs.apply(execs.size() - 1)
    values, planned = store.executionMetrics(last.executionId()), last.metrics()
    seen = {}
    for i in range(planned.size()):
        m = planned.apply(i)
        v = values.get(m.accumulatorId())
        if v.isDefined():
            seen[m.accumulatorId()] = (m.name(), v.get())
    out = {}
    for name, text in seen.values():
        out.setdefault(name, []).append(text.strip().split("\n")[-1])
    return out

---

## 1. The four representations

Section 3.8 describes a pipeline of four successive plans: the **unresolved logical plan**
parsed from the query, the **logical plan** resolved against the catalog, the **optimized
logical plan** produced by the rule rewrites, and the **physical plan** selected by cost.
`explain(mode="extended")` prints all four.

The query below is written badly on purpose. It projects first and filters second, it asks for
a column it never uses, and it computes an arithmetic constant per row.

In [3]:
trips = spark.createDataFrame(
    [(1, "CRD", 2.4, 12.5, 2025), (2, "CSH", 0.8,  6.0, 2025),
     (3, "CRD", 7.1, 31.0, 2024), (4, "CSH", 3.3, 15.5, 2025),
     (5, "CRD", 0.4,  4.5, 2024)],
    ["trip_id", "payment_type", "miles", "fare", "year"])

badly_written = (trips
                 .select("trip_id", "payment_type", "miles", "fare", "year")   # every column
                 .withColumn("fare_cents", F.col("fare") * (F.lit(10) * F.lit(10)))
                 .where(F.col("year") == 2025)                                 # filter last
                 .select("payment_type", "fare_cents"))                        # two columns

badly_written.explain(mode="extended")

== Parsed Logical Plan ==
'Project ['payment_type, 'fare_cents]
+- Filter (year#4L = cast(2025 as bigint))
   +- Project [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L, (fare#3 * cast((10 * 10) as double)) AS fare_cents#5]
      +- Project [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L]
         +- LogicalRDD [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L], false

== Analyzed Logical Plan ==
payment_type: string, fare_cents: double
Project [payment_type#1, fare_cents#5]
+- Filter (year#4L = cast(2025 as bigint))
   +- Project [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L, (fare#3 * cast((10 * 10) as double)) AS fare_cents#5]
      +- Project [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L]
         +- LogicalRDD [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L], false

== Optimized Logical Plan ==
Project [payment_type#1, (fare#3 * 100.0) AS fare_cents#5]
+- Filter (isnotnull(year#4L) AND (year#4L = 2025))
   +- LogicalRDD [trip_id#0L, payment_ty

Read the four blocks against each other.

* **Parsed** — the tree as written, names unchecked. `'Project`, `'Filter` with a leading quote
  mark unresolved attributes.
* **Analyzed** — the same tree with every name bound to a column of known type.
* **Optimized** — this is where the work happens. Three rewrites from section 3.8 are visible
  at once. The **filter has moved below the projections**, so rows are discarded before any
  arithmetic is done on them (*predicate pushdown*). The columns never used downstream have
  **disappeared from the plan** (*projection pruning*). And `fare * (10 * 10)` has become
  `fare * 100` (*constant folding*) — the sub-expression that does not depend on the data was
  evaluated once, at planning time, rather than uselessly for every row.
* **Physical** — the executable form, with a `Scan` at the bottom and the operators fused into
  whole-stage-generated code.

The badly written query and a well written one therefore *converge*. That is the entire point
of the chapter: Spark understands what the query means, so it is free to rewrite it.

In [4]:
well_written = (trips
                .where(F.col("year") == 2025)
                .select("payment_type", (F.col("fare") * 100).alias("fare_cents")))

a = badly_written._jdf.queryExecution().optimizedPlan().toString()
b = well_written._jdf.queryExecution().optimizedPlan().toString()
print("optimized plans identical:", normalise(a) == normalise(b))
print(a)

optimized plans identical: True
Project [payment_type#1, (fare#3 * 100.0) AS fare_cents#5]
+- Filter (isnotnull(year#4L) AND (year#4L = 2025))
   +- LogicalRDD [trip_id#0L, payment_type#1, miles#2, fare#3, year#4L], false



None of these rewrites is available to an RDD program, for the reason section 3.1 gives:
Spark cannot see inside `rdd.map(f)` and therefore cannot know that a filter is safe to move or
a column safe to omit.

---

## 2. SQL and the method chain are one query

Section 3.5 says a SQL string and the equivalent chain of method calls are parsed into the same
internal representation, handed to the same optimizer, and therefore execute at identical speed.
That is a testable claim, and this is the test (Exercise 6).

In [5]:
trips.createOrReplaceTempView("trips")

sql_form = spark.sql("""
    SELECT payment_type, AVG(fare) AS avg_fare
    FROM trips
    WHERE miles > 2
    GROUP BY payment_type
    ORDER BY avg_fare DESC
""")

method_form = (trips
               .where(F.col("miles") > 2)
               .groupBy("payment_type")
               .agg(F.avg("fare").alias("avg_fare"))
               .orderBy(F.desc("avg_fare")))

sql_plan    = sql_form._jdf.queryExecution().executedPlan().toString()
method_plan = method_form._jdf.queryExecution().executedPlan().toString()

# Catalyst numbers every expression it creates (`fare#3`), and two separately built
# DataFrames get different numbers for the same column.  Those identifiers are the only
# difference between these two plans, so they are normalised away before comparing.
print("physical plans identical:", normalise(sql_plan) == normalise(method_plan))
print()
print(sql_plan)

physical plans identical: True

AdaptiveSparkPlan isFinalPlan=false
+- Sort [avg_fare#7 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(avg_fare#7 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=41]
      +- HashAggregate(keys=[payment_type#1], functions=[avg(fare#3)], output=[payment_type#1, avg_fare#7])
         +- Exchange hashpartitioning(payment_type#1, 200), ENSURE_REQUIREMENTS, [plan_id=38]
            +- HashAggregate(keys=[payment_type#1], functions=[partial_avg(fare#3)], output=[payment_type#1, sum#18, count#19L])
               +- Project [payment_type#1, fare#3]
                  +- Filter (isnotnull(miles#2) AND (miles#2 > 2.0))
                     +- Scan ExistingRDD[trip_id#0L,payment_type#1,miles#2,fare#3,year#4L]



In [6]:
sql_form.show()
method_form.show()

+------------+--------+
|payment_type|avg_fare|
+------------+--------+
|         CRD|   21.75|
|         CSH|    15.5|
+------------+--------+

+------------+--------+
|payment_type|avg_fare|
+------------+--------+
|         CRD|   21.75|
|         CSH|    15.5|
+------------+--------+



The two plans are the same string. Not similar — the same. The choice between the surfaces is
therefore one of readability alone: a query dominated by joins and grouping is often clearer as
SQL, a long pipeline of derived columns clearer as method calls, and a program may move between
them from one line to the next.

---

## 3. The join strategy is chosen by size

Section 3.9.2 lists four physical strategies and says the choice is made on the estimated sizes
of the inputs, governed by one configuration parameter. Here are the defaults Spark 4 actually
uses.

In [7]:
DEFAULTS = ["spark.sql.autoBroadcastJoinThreshold",
            "spark.sql.adaptive.enabled",
            "spark.sql.shuffle.partitions",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes",
            "spark.sql.adaptive.coalescePartitions.enabled",
            "spark.sql.adaptive.skewJoin.enabled",
            "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
            "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes"]

for key in DEFAULTS:
    value = spark.conf.get(key)
    pretty = ""
    if value.endswith("b") and value[:-1].isdigit():
        pretty = f"  ({int(value[:-1]) / 1024**2:.0f} MB)"
    print(f"{key:62s} = {value}{pretty}")

spark.sql.autoBroadcastJoinThreshold                           = 10485760b  (10 MB)
spark.sql.adaptive.enabled                                     = true
spark.sql.shuffle.partitions                                   = 200
spark.sql.adaptive.advisoryPartitionSizeInBytes                = 67108864b  (64 MB)
spark.sql.adaptive.coalescePartitions.enabled                  = true
spark.sql.adaptive.skewJoin.enabled                            = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor              = 5.0
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes    = 268435456b  (256 MB)


Every figure the chapter quotes is there: a 10 MB broadcast threshold, AQE on by default, 200
shuffle partitions, a 64 MB advisory partition size, a skew factor of 5 and a 256 MB skew
threshold.

Now two tables, written to Parquet so that Spark must estimate their sizes from the files
rather than from an in-memory catalog — which is how it works in practice, and where the
estimates go wrong.

In [8]:
BIG   = os.path.join(WORK, "big")       # ~4 M rows
SMALL = os.path.join(WORK, "small")     # ~2 M rows, wide enough to exceed 10 MB
TINY  = os.path.join(WORK, "tiny")      # 1 000 rows, comfortably under 10 MB

if not os.path.exists(TINY):
    (spark.range(4_000_000)
          .withColumn("k", (F.rand(1) * 1000).cast("int"))
          .withColumn("amount", F.rand(2) * 100)
          .write.mode("overwrite").parquet(BIG))
    (spark.range(2_000_000)
          .withColumn("k", (F.rand(5) * 1000).cast("int"))
          .withColumn("w", F.rand(6))
          .withColumn("pad", F.md5(F.col("id").cast("string")))
          .write.mode("overwrite").parquet(SMALL))
    (spark.range(1000)
          .withColumnRenamed("id", "k")
          .withColumn("region", F.concat(F.lit("R"), (F.col("k") % 5).cast("string")))
          .write.mode("overwrite").parquet(TINY))

def mb(path):
    return round(sum(os.path.getsize(os.path.join(r, f))
                     for r, _, fs in os.walk(path) for f in fs) / 1024**2, 2)

big, small, tiny = (spark.read.parquet(p) for p in (BIG, SMALL, TINY))
for name, path in [("big", BIG), ("small", SMALL), ("tiny", TINY)]:
    print(f"{name:6s} {mb(path):8.2f} MB on disk")

big       51.09 MB on disk
small     88.21 MB on disk
tiny       0.02 MB on disk


In [9]:
def strategy_of(df):
    """The join operator Spark chose, pulled out of the physical plan."""
    plan = df._jdf.queryExecution().executedPlan().toString()
    for line in plan.split("\n"):
        for name in ("BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin",
                     "BroadcastNestedLoopJoin", "CartesianProduct"):
            if name in line:
                return name
    return "no join operator found"

print("big JOIN tiny  (one side under 10 MB) ->", strategy_of(big.join(tiny,  on="k")))
print("big JOIN small (both sides over 10 MB) ->", strategy_of(big.join(small, on="k")))

big JOIN tiny  (one side under 10 MB) -> BroadcastHashJoin
big JOIN small (both sides over 10 MB) -> SortMergeJoin


Exactly as section 3.9.2 predicts: a side estimated below the threshold is broadcast, so the
large table is joined in place and never moved across the network; two large sides fall back to
a sort-merge join, which shuffles both.

**When the estimate is wrong.** The weak point is the estimate itself. For files read directly
Spark falls back on the size on disk, which knows nothing about a filter applied afterwards.
Here `small` is filtered down to a few thousand rows, and the planner cannot tell.

In [10]:
selective = small.where(F.col("w") < 0.001).select("k")     # ~2 000 of 2 000 000 rows
print("rows that survive the filter:", selective.count())
print("strategy chosen from the file size:", strategy_of(big.join(selective, on="k")))

# The broadcast() hint overrides the estimate.  This is the one line Exercise 5(a) asks for.
from pyspark.sql.functions import broadcast
print("strategy with a broadcast() hint:  ", strategy_of(big.join(broadcast(selective), on="k")))

rows that survive the filter: 2056
strategy chosen from the file size: SortMergeJoin
strategy with a broadcast() hint:   BroadcastHashJoin


The same instruction is available in SQL as a hint comment, and the complementary hints
`MERGE`, `SHUFFLE_HASH` and `SHUFFLE_REPLICATE_NL` force the remaining strategies.

In [11]:
big.createOrReplaceTempView("big")
tiny.createOrReplaceTempView("tiny")

hinted = spark.sql("SELECT /*+ BROADCAST(tiny) */ COUNT(*) FROM big JOIN tiny USING (k)")
merged = spark.sql("SELECT /*+ MERGE(tiny) */ COUNT(*) FROM big JOIN tiny USING (k)")
print("with /*+ BROADCAST(tiny) */ ->", strategy_of(hinted))
print("with /*+ MERGE(tiny) */     ->", strategy_of(merged))

with /*+ BROADCAST(tiny) */ -> BroadcastHashJoin
with /*+ MERGE(tiny) */     -> SortMergeJoin


Hints should be used sparingly and deliberately, to correct a mis-estimate that has actually
been observed, because a hint that is right for today's data becomes wrong once the tables grow.
Which is the natural cue for the next section: Spark can often fix the mis-estimate itself,
without a hint, by waiting until it knows the truth.

---

## 4. Adaptive Query Execution

Everything Catalyst does, it does *before* the query runs, from estimates. AQE re-optimizes
*while* it runs, using the true sizes of intermediate data observed after each shuffle. The
information is free: Spark already writes every shuffle to disk and records the size of every
partition.

### 4.1 Coalescing shuffle partitions (Exercise 9)

`spark.sql.shuffle.partitions` fixes the post-shuffle partition count in advance, at 200. One
fixed number cannot suit every query: for a small result it produces two hundred nearly empty
partitions and two hundred corresponding scheduling overheads. AQE waits until the true
post-shuffle size is known and then coalesces adjacent small partitions until each reaches
`advisoryPartitionSizeInBytes`.

In [12]:
rows = []
for aqe in [False, True]:
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    q = big.groupBy("k").agg(F.avg("amount"))
    t0 = time.time()
    q.collect()
    seconds = time.time() - t0
    m = sql_metrics()
    rows.append({"AQE": aqe,
                 "seconds": round(seconds, 2),
                 "partitions requested": m.get("number of partitions", ["-"])[0],
                 "partitions after coalescing": m.get("number of coalesced partitions", ["-"])[0],
                 "empty partitions": m.get("number of empty partitions", ["-"])[0]})

spark.conf.set("spark.sql.adaptive.enabled", True)          # back to the default
print(pd.DataFrame(rows).to_string(index=False))

  AQE  seconds partitions requested partitions after coalescing empty partitions
False     0.64                  200                           -                -
 True     0.50                  200                           1                2


With AQE off, the aggregation's shuffle produces 200 partitions and 200 tasks whatever the
result's size. With it on, the shuffle still *writes* 200 partitions — the write side is fixed
in advance — but the read side coalesces them down to a handful of tasks sized to the data. The
`AQEShuffleRead coalesced` node in the final plan is where that happens.

In [13]:
coalescing = big.groupBy("k").agg(F.avg("amount"))
coalescing.collect()                     # the final plan exists only after the query has run
print(final_plan(coalescing)[:700])

AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- *(2) HashAggregate(keys=[k#53], functions=[avg(amount#54)], output=[k#53, avg(amount)#105])
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 0
            +- Exchange hashpartitioning(k#53, 200), ENSURE_REQUIREMENTS, [plan_id=631]
               +- *(1) HashAggregate(keys=[k#53], functions=[partial_avg(amount#54)], output=[k#53, sum#108, count#109L])
                  +- *(1) ColumnarToRow
                     +- FileScan parquet [k#53,amount#54] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch03-


### 4.2 Converting a sort-merge join to a broadcast join

This is the mis-estimate of section 3 above, fixed without a hint. The static plan sees a 90 MB
file on one side and plans a sort-merge join. After the shuffle, AQE observes that the filtered
side is a few thousand rows, and converts the join then and there — eliminating a shuffle the
static plan would have performed for nothing.

In [14]:
converted = big.join(selective, on="k").agg(F.count("*"))

print("=== plan BEFORE execution (what Catalyst decided from estimates) ===")
converted.explain()

=== plan BEFORE execution (what Catalyst decided from estimates) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[count(1)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=692]
      +- HashAggregate(keys=[], functions=[partial_count(1)])
         +- Project
            +- SortMergeJoin [k#53], [k#56], Inner
               :- Sort [k#53 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(k#53, 200), ENSURE_REQUIREMENTS, [plan_id=684]
               :     +- Filter isnotnull(k#53)
               :        +- FileScan parquet [k#53] Batched: true, DataFilters: [isnotnull(k#53)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch03-catal..., PartitionFilters: [], PushedFilters: [IsNotNull(k)], ReadSchema: struct<k:int>
               +- Sort [k#56 ASC NULLS FIRST], false, 0
                  +- Exchange hashpartitioning(k#56, 200), ENSURE_RE

In [15]:
converted.collect()
print("=== plan AFTER execution (what AQE actually ran) ===")
print(final_plan(converted)[:700])

=== plan AFTER execution (what AQE actually ran) ===
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- *(4) HashAggregate(keys=[], functions=[count(1)], output=[count(1)#118L])
      +- ShuffleQueryStage 3
         +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=851]
            +- *(3) HashAggregate(keys=[], functions=[partial_count(1)], output=[count#120L])
               +- *(3) Project
                  +- *(3) BroadcastHashJoin [k#53], [k#56], Inner, BuildRight, false, false
                     :- AQEShuffleRead local
                     :  +- ShuffleQueryStage 0
                     :     +- Exchange hashpartitioning(k#53, 200), ENSURE_REQUIREMENTS, [plan_id=723]
                     :       


`SortMergeJoin` before, `BroadcastHashJoin` after. Note also `AQEShuffleRead local` on the
large side: having decided to broadcast, AQE reads the already-written shuffle files *locally*
rather than fetching them across the network, because each task now needs only its own data.

This is why `explain()` alone can mislead on an AQE-enabled cluster. It prints the plan as it
stood *before* execution, marked `isFinalPlan=false`. The plan that ran is the one printed
after the query completes, marked `isFinalPlan=true`.

---

## 5. The "ninety-nine percent complete" problem (Exercise 8)

Now the pathology the course has returned to since its opening pages. A stage cannot complete
until its slowest task completes, and a job that sits at "ninety-nine percent" for an hour is
almost invariably waiting on one task that was given far more data than its peers.

The cause is **data skew**: a join key whose values are distributed very unevenly, so that one
key — and therefore the one partition that key hashes to — holds a large fraction of all the
records.

Here it is built on purpose: seventy percent of eight million rows carry the country code
`US`.

In [16]:
FACTS = os.path.join(WORK, "facts")
DIMS  = os.path.join(WORK, "dims")

if not os.path.exists(DIMS):
    (spark.range(8_000_000)
          .withColumn("country", F.when(F.rand(7) < 0.7, F.lit("US"))
                                  .otherwise(F.concat(F.lit("C"), (F.rand(11) * 40).cast("int").cast("string"))))
          .withColumn("amount", F.rand(3) * 100)
          .withColumn("pad", F.md5(F.col("id").cast("string")))
          .write.mode("overwrite").parquet(FACTS))
    (spark.range(41)
          .withColumn("country", F.when(F.col("id") == 0, F.lit("US"))
                                  .otherwise(F.concat(F.lit("C"), (F.col("id") - 1).cast("string"))))
          .withColumn("region", F.concat(F.lit("R"), (F.col("id") % 5).cast("string")))
          .drop("id")
          .write.mode("overwrite").parquet(DIMS))

facts = spark.read.parquet(FACTS)
dims  = spark.read.parquet(DIMS)

facts.groupBy("country").count().orderBy(F.desc("count")).show(6)

+-------+-------+
|country|  count|
+-------+-------+
|     US|5600036|
|    C32|  60523|
|    C13|  60500|
|     C4|  60478|
|    C11|  60316|
|    C34|  60256|
+-------+-------+
only showing top 6 rows


One key holds 5.6 million of 8 million rows; the next holds sixty thousand. Every row with
`country = "US"` hashes to the same shuffle partition, so one task receives ninety times the
work of its peers, and the whole stage waits for it.

The join is forced to sort-merge below, by setting the broadcast threshold to `-1`. On this
small dimension table Spark would otherwise broadcast and the skew would never arise — which is
worth noting as the first remedy to reach for whenever the small side permits it.

The skew thresholds are also scaled down. Their defaults are 256 MB and a factor of five; a
laptop-sized dataset never reaches 256 MB per partition, so the demonstration would show
nothing at the default. **This is a change of scale, not of mechanism** — the rule being applied
is exactly the one section 3.10.1 states.

In [17]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)              # force sort-merge
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", 4 * 1024 * 1024)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", 4 * 1024 * 1024)
# skewedPartitionFactor stays at its default of 5

def skewed_join():
    return facts.join(dims, on="country").groupBy("region").agg(F.sum("amount").alias("total"))

skew_rows = []
for enabled in [False, True]:
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", enabled)
    q = skewed_join()
    t0 = time.time()
    q.collect()
    seconds = time.time() - t0
    plan = final_plan(q)
    skew_rows.append({"skewJoin.enabled": enabled,
                      "seconds": round(seconds, 2),
                      "join operator": "SortMergeJoin(skew=true)" if "skew=true" in plan else "SortMergeJoin",
                      "shuffle read": "coalesced and skewed" if "coalesced and skewed" in plan
                                       else "coalesced"})
    globals()[f"plan_{enabled}"] = plan

print(pd.DataFrame(skew_rows).to_string(index=False))

 skewJoin.enabled  seconds            join operator         shuffle read
            False     1.39            SortMergeJoin            coalesced
             True     0.48 SortMergeJoin(skew=true) coalesced and skewed


The plans say what happened, in two words each.

In [18]:
def join_lines(plan):
    return "\n".join(l for l in plan.split("\n")
                     if "SortMergeJoin" in l or "AQEShuffleRead" in l)

print("=== skewJoin disabled ===")
print(join_lines(plan_False))
print()
print("=== skewJoin enabled ===")
print(join_lines(plan_True))

=== skewJoin disabled ===
      +- AQEShuffleRead coalesced
                     +- *(5) SortMergeJoin [country#129], [country#132], Inner
                        :  +- AQEShuffleRead coalesced
                           +- AQEShuffleRead coalesced

=== skewJoin enabled ===
      +- AQEShuffleRead coalesced
                     +- *(5) SortMergeJoin(skew=true) [country#129], [country#132], Inner
                        :  +- AQEShuffleRead coalesced and skewed
                           +- AQEShuffleRead coalesced


`SortMergeJoin(skew=true)` and `AQEShuffleRead coalesced and skewed`. AQE detected the
oversized partition from the shuffle statistics — larger than five times the median *and* over
the absolute floor — split it into several sub-partitions, and replicated the matching rows
from the dimension side to each of them, so that the work that would have fallen to one
overloaded task is spread across many tasks of roughly equal size.

The imbalance itself is visible directly. `spark_partition_id()` names the partition each row
landed in, so grouping by it counts the rows each task had to process.

In [19]:
# AQE would coalesce the shuffle partitions before `spark_partition_id` could see them,
# so this one measurement runs with AQE off: these are the raw 200 hash partitions, which
# is what a task in the join stage is actually handed.
spark.conf.set("spark.sql.adaptive.enabled", False)

plain = (facts.join(dims, on="country")
              .withColumn("pid", F.spark_partition_id())
              .groupBy("pid").count().orderBy(F.desc("count")))
sizes = [r["count"] for r in plain.collect()]
print(f"partitions holding data: {len(sizes)}")
print(f"largest task:  {max(sizes):>9,d} rows")
print(f"median task:   {int(pd.Series(sizes).median()):>9,d} rows")
print(f"ratio:         {max(sizes) / pd.Series(sizes).median():>9.1f}x the median")
plain.show(5)
spark.conf.set("spark.sql.adaptive.enabled", True)

partitions holding data: 39
largest task:  5,659,729 rows
median task:      59,948 rows
ratio:              94.4x the median


+---+-------+
|pid|  count|
+---+-------+
| 92|5659729|
| 79| 120305|
|157|  60523|
|151|  60500|
| 57|  60478|
+---+-------+
only showing top 5 rows


That ratio is the straggler, quantified: one task with a load many times the median, and every
other task finished and idle while it grinds on.

**Where the automatic cure does not reach.** AQE splits a skewed *join* partition because it can
replicate the other side's matching rows to each split. A skewed **aggregation** has no other
side to replicate, so the same trick does not apply, and the classical manual remedy is
**salting**: append a small random value to the offending key so its records scatter across many
partitions, aggregate over the salted keys, then strip the salt and aggregate the partial
results. That is the answer to Exercise 8(c), and it is the same idea AQE applies to joins on
the programmer's behalf — break one enormous group into many small ones.

In [20]:
SALT = 16
salted = (facts
          .withColumn("salt", (F.rand(99) * SALT).cast("int"))
          .groupBy("country", "salt").agg(F.sum("amount").alias("part"))   # many small groups
          .groupBy("country").agg(F.sum("part").alias("total")))           # then combine

direct = facts.groupBy("country").agg(F.sum("amount").alias("total"))

# Same answer, reached by a differently shaped computation.
check = (direct.alias("d").join(salted.alias("s"), on="country")
               .select(F.max(F.abs(F.col("d.total") - F.col("s.total"))).alias("max_difference")))
check.show()
salted.orderBy(F.desc("total")).show(5)
# The difference is not exactly zero, and should not be: floating-point addition is not
# associative, so summing the same numbers in a different order changes the last bits.

+--------------------+
|      max_difference|
+--------------------+
|1.192092895507812...|
+--------------------+



+-------+-------------------+
|country|              total|
+-------+-------------------+
|     US|2.800148575639379E8|
|     C4| 3034935.1934281136|
|    C32|  3025028.052805429|
|     C8| 3024368.9442406585|
|    C34| 3015506.3800634528|
+-------+-------------------+
only showing top 5 rows


In [21]:
# Restore every configuration this notebook changed.
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.sql.adaptive.advisoryPartitionSizeInBytes")
spark.conf.unset("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes")
spark.conf.unset("spark.sql.adaptive.skewJoin.enabled")
spark.conf.unset("spark.sql.adaptive.enabled")
for key in DEFAULTS:
    print(f"{key:62s} = {spark.conf.get(key)}")

spark.sql.autoBroadcastJoinThreshold                           = 10485760b
spark.sql.adaptive.enabled                                     = true
spark.sql.shuffle.partitions                                   = 200
spark.sql.adaptive.advisoryPartitionSizeInBytes                = 67108864b
spark.sql.adaptive.coalescePartitions.enabled                  = true
spark.sql.adaptive.skewJoin.enabled                            = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor              = 5.0
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes    = 268435456b


## Conclusion

Four claims from the chapter, each now checked rather than believed:

1. **A query is rewritten before it runs.** A deliberately badly written query and a carefully
   written one optimized to *the same* logical plan — filter moved below the projection, unused
   columns dropped, constant sub-expression folded.
2. **SQL and the DataFrame API are one engine.** The two physical plans compared equal as
   strings. The choice between the surfaces is readability alone.
3. **The join strategy follows the estimated size**, with a 10 MB default threshold; and where
   the estimate is wrong — a heavy filter the planner cannot see through — either a
   `broadcast()` hint or AQE will correct it, and AQE is preferable because it corrects itself
   as the data changes.
4. **AQE revises the plan mid-flight**: 200 shuffle partitions coalesced down to the size of the
   result, a sort-merge join converted to a broadcast join once the shuffle told the truth, and
   a skewed partition split into balanced pieces so that no single task holds up its stage.

Two habits to take away.

**Read the plan, and read the right one.** `explain()` prints what Catalyst decided
(`isFinalPlan=false`); the executed plan, after the query finishes, prints what actually ran.
On an AQE-enabled cluster those can differ in the operator that matters most.

**Skew is a property of the data, not of the code.** The join above is unremarkable; it is the
seventy percent of rows sharing one key that turns it into an hour-long stage. AQE cures the
join case automatically and the aggregation case not at all, which is why salting is still worth
knowing.